In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import pandas as pd

# 1. Inisialisasi
client = Client("IRIS")
CATALOG_PATH = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"

# 2. Ambil 1 contoh event terbaru (misal dari baris pertama)
df = pd.read_csv(CATALOG_PATH)
row = df.iloc[0]
event_time = UTCDateTime(pd.to_datetime(f"{row['Date']} {row['Time (UTC)']}"))

print(f"🔍 Mencari semua stasiun yang punya data untuk event: {row['Event ID']}")

try:
    # Kita minta METADATA stasiun, bukan waveform-nya dulu
    # Kita cari di seluruh Indonesia (Lat: -11 s/d 6, Lon: 95 s/d 141)
    inventory = client.get_stations(
        network="*", 
        station="*", 
        starttime=event_time - 30, 
        endtime=event_time + 90,
        minlatitude=-11.0, maxlatitude=6.0,
        minlongitude=95.0, maxlongitude=141.0,
        level="channel"
    )
    
    print("\n✅ Stasiun yang ditemukan di server IRIS untuk waktu tersebut:")
    for net in inventory:
        for sta in net:
            print(f"- Network: {net.code} | Station: {sta.code} | Channels: {[c.code for c in sta]}")

except Exception as e:
    print(f"❌ Error saat mengecek: {e}")

In [ ]:
import os
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm
import time

# --- CONFIG ---
CATALOG_PATH = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"
SAVE_DIR = r"D:\indonesia_output\waveforms"

clients = [Client("IRIS"), Client("GFZ")]
STATION_CODES = ["BJI", "BBJI", "GSI", "KAPI", "LBMI", "LUWI", "PMBI", "PPI", "SANI", 
                 "TOLI", "TNTI", "JAGI", "PLAI", "UGM", "BOAB", "FAU", "MSI", "BKB"]

MIN_FILE_SIZE = 6144 # 6 KB

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def download_final_resilient():
    df = pd.read_csv(CATALOG_PATH)
    df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)
    df_filtered = df[(df['Magnitude'] >= 4.0) & (df['Date_dt'].dt.year >= 2020)]
    
    print(f"🚀 Memulai Audit & Unduh: Target data sehat > 6KB")

    for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
        event_id = row['Event ID']
        try:
            event_time = UTCDateTime(pd.to_datetime(f"{row['Date']} {row['Time (UTC)']}"))
        except: continue
        
        for sta in STATION_CODES:
            file_path = os.path.join(SAVE_DIR, f"{sta}_{event_id}.mseed")
            
            # Jika sudah ada dan SEHAT (>6KB), lewati
            if os.path.exists(file_path) and os.path.getsize(file_path) > MIN_FILE_SIZE:
                continue
            
            # Jika ada tapi "Sakit" (<=5KB), hapus agar bisa coba lagi
            if os.path.exists(file_path):
                os.remove(file_path)
            
            found = False
            for cl in clients:
                if found: break
                # Coba 2 variasi Location Code: "*" (semua) dan "--" (kosong/standard)
                for loc in ["*", "--"]:
                    if found: break
                    try:
                        st = cl.get_waveforms(network="*", station=sta, location=loc, 
                                             channel="*Z", starttime=event_time-30, endtime=event_time+90)
                        if len(st) > 0:
                            st[0:1].write(file_path, format="MSEED")
                            if os.path.getsize(file_path) > MIN_FILE_SIZE:
                                found = True
                                break
                    except:
                        continue

if __name__ == "__main__":
    download_final_resilient()

In [ ]:
import os
import glob

# --- CONFIG ---
# Gunakan path Windows Bapak
SAVE_DIR = r"D:\indonesia_output\waveforms"
# Kita naikkan ambang batas ke 6KB untuk memastikan file kosong/header saja terhapus
SIZE_THRESHOLD_KB = 6 

def clean_junk_waveforms():
    print(f"🧹 Memulai pembersihan file di bawah {SIZE_THRESHOLD_KB}KB di: {SAVE_DIR}")
    
    # Ambil semua file .mseed
    files = glob.glob(os.path.join(SAVE_DIR, "*.mseed"))
    
    deleted_count = 0
    kept_count = 0
    
    for f in files:
        # Ambil ukuran file dalam KB
        file_size_kb = os.path.getsize(f) / 1024
        
        if file_size_kb < SIZE_THRESHOLD_KB:
            try:
                os.remove(f)
                deleted_count += 1
            except Exception as e:
                print(f"❌ Gagal menghapus {os.path.basename(f)}: {e}")
        else:
            kept_count += 1
            
    print("-" * 30)
    print(f"✅ Pembersihan Selesai!")
    print(f"🗑️ File dihapus (Ukuran < {SIZE_THRESHOLD_KB}KB): {deleted_count}")
    print(f"📦 File sehat yang tersisa: {kept_count}")
    print("-" * 30)

if __name__ == "__main__":
    clean_junk_waveforms()

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import pandas as pd

# 1. Inisialisasi Dua Client
clients = {
    "IRIS (Amerika)": Client("IRIS"),
    "GFZ (Jerman)": Client("GFZ")
}

CATALOG_PATH = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"

# 2. Ambil 1 contoh event
df = pd.read_csv(CATALOG_PATH)
row = df.iloc[0]
event_time = UTCDateTime(pd.to_datetime(f"{row['Date']} {row['Time (UTC)']}"))

print(f"🔍 Audit Metadata untuk Event ID: {row['Event ID']}")
print(f"⏰ Waktu Kejadian: {event_time}\n")

# 3. Iterasi pengecekan ke setiap Client
for name, client in clients.items():
    print(f"--- Mengecek di Server {name} ---")
    try:
        inventory = client.get_stations(
            network="*", 
            station="*", 
            starttime=event_time - 30, 
            endtime=event_time + 90,
            minlatitude=-11.0, maxlatitude=6.0,
            minlongitude=95.0, maxlongitude=141.0,
            level="channel"
        )
        
        found_count = 0
        for net in inventory:
            for sta in net:
                channels = list(set([c.code for c in sta])) # Unikkan kode channel
                print(f"✅ Net: {net.code} | Sta: {sta.code} | Ch: {channels}")
                found_count += 1
        
        if found_count == 0:
            print("⚠️ Tidak ada stasiun ditemukan di server ini.")
            
    except Exception as e:
        print(f"❌ Server {name} tidak merespons atau error: {e}")
    print("\n")

In [ ]:
import os
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm
import time

# --- CONFIG PATH WINDOWS ---
CATALOG_PATH = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"
SAVE_DIR = r"D:\indonesia_output\waveforms"

# Inisialisasi dua pintu utama (Global & Regional)
client_iris = Client("IRIS", timeout=30)
client_gfz = Client("GFZ", timeout=30)

# Daftar 18 Stasiun Target
STATION_CODES = [
    "BJI", "BBJI", "GSI", "KAPI", "LBMI", "LUWI", "PMBI", "PPI", "SANI", 
    "TOLI", "TNTI", "JAGI", "PLAI", "UGM", "BOAB", "FAU", "MSI", "BKB"
]

# Ambang batas ukuran file untuk 3-Komponen (E, N, Z)
# File 3-komponen yang berisi biasanya > 50KB. 10KB adalah batas aman minimal.
MIN_SIZE = 10240  # 10 KB

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def panen_raya_3_komponen():
    print(f"📖 Membaca katalog: {CATALOG_PATH}")
    try:
        df = pd.read_csv(CATALOG_PATH)
    except Exception as e:
        print(f"❌ Error: Gagal membaca CSV. Cek path! {e}")
        return

    # Filter: Tahun 2020-2024 & Magnitudo >= 4.0
    df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)
    df_filtered = df[(df['Magnitude'] >= 4.0) & (df['Date_dt'].dt.year >= 2020)]
    
    total_events = len(df_filtered)
    print(f"🚜 Memulai Panen Raya 3-Komponen (E, N, Z)")
    print(f"📊 Target: {total_events} Event x {len(STATION_CODES)} Stasiun")

    for _, row in tqdm(df_filtered.iterrows(), total=total_events):
        event_id = row['Event ID']
        try:
            # Konversi waktu BMKG ke ObsPy UTCDateTime
            t_str = f"{row['Date']} {row['Time (UTC)']}"
            event_time = UTCDateTime(pd.to_datetime(t_str))
        except:
            continue
        
        for sta in STATION_CODES:
            file_name = f"{sta}_{event_id}.mseed"
            file_path = os.path.join(SAVE_DIR, file_name)
            
            # 1. FITUR RESUME: Jika file sudah ada dan ukurannya sehat, lewati.
            if os.path.exists(file_path) and os.path.getsize(file_path) > MIN_SIZE:
                continue

            # 2. PROSES UNDUH DUAL-GATE (IRIS & GFZ)
            success = False
            # Coba IRIS dulu, jika gagal coba GFZ
            for cl_name, cl in [("IRIS", client_iris), ("GFZ", client_gfz)]:
                if success: break
                try:
                    # Ambil channel broadband BH?, HH?, EH? (E, N, Z)
                    # Jika ingin sapu bersih gunakan channel="*"
                    st = cl.get_waveforms(
                        network="*", 
                        station=sta, 
                        location="*", 
                        channel="BH?,HH?,EH?,*?", 
                        starttime=event_time - 30, 
                        endtime=event_time + 90
                    )
                    
                    if len(st) > 0:
                        # Pilih hanya trace dengan sampling rate tertinggi (biasanya 100Hz)
                        st.sort(keys=['sampling_rate'], reverse=True)
                        
                        # Ambil 3 komponen pertama yang unik (E, N, Z)
                        # st.write akan menyimpan semua trace yang ada di objek stream
                        st.write(file_path, format="MSEED")
                        
                        # Cek apakah file hasil unduhan ini "gemuk" (>10KB)
                        if os.path.getsize(file_path) > MIN_SIZE:
                            success = True 
                        else:
                            # Jika hasil unduhan tetap kecil (zombie file), hapus agar tidak mengotori
                            if os.path.exists(file_path):
                                os.remove(file_path)
                except:
                    continue # Lanjut coba client berikutnya

if __name__ == "__main__":
    panen_raya_3_komponen()

In [ ]:

import json
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm
import os

# --- CONFIG ---
CATALOG_PATH = r"D:\indonesia_output\Indonesian_Earthquake_Catalog_BMKG_1998_2024\BMKG_Earthquake_Catalog.csv"
OUTPUT_JSON = r"D:\indonesia_output\radar_inventory_3ch_detailed.json"
STATION_CODES = ["BJI", "BBJI", "GSI", "KAPI", "LBMI", "LUWI", "PMBI", "PPI", "SANI", 
                 "TOLI", "TNTI", "JAGI", "PLAI", "UGM", "BOAB", "FAU", "MSI", "BKB"]

client_iris = Client("IRIS", timeout=30)

def run_resumable_radar():
    print(f"📖 Membaca katalog...")
    df = pd.read_csv(CATALOG_PATH)
    df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)
    df_filtered = df[(df['Magnitude'] >= 4.0) & (df['Date_dt'].dt.year >= 2020)].copy()
    
    # --- FITUR RESUME ---
    radar_results = []
    processed_ids = set()
    
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r') as f:
            try:
                radar_results = json.load(f)
                processed_ids = {entry['Event_ID'] for entry in radar_results}
                print(f"🔄 Melanjutkan proses. {len(processed_ids)} event sudah pernah diproses sebelumnya.")
            except:
                print("⚠️ File JSON lama korup atau kosong, memulai dari awal.")
    # --------------------

    print(f"📡 Menjalankan Radar 3-Komponen (Resumable)...")

    for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
        event_id = row['Event ID']
        
        # Cek apakah ID ini sudah ada di hasil sebelumnya
        if event_id in processed_ids:
            continue
            
        try:
            event_time = UTCDateTime(pd.to_datetime(f"{row['Date']} {row['Time (UTC)']}"))
            
            inv = client_iris.get_stations(
                starttime=event_time-5, endtime=event_time+5,
                network="*", station=",".join(STATION_CODES), 
                level="channel"
            )
            
            valid_stations_info = []
            for net in inv:
                for sta in net:
                    channels = [c.code for c in sta]
                    broadband_ch = [c for c in channels if c[0:2] in ['BH', 'HH', 'EH']]
                    
                    if len(set(broadband_ch)) >= 3:
                        valid_stations_info.append({
                            "station": sta.code,
                            "network": net.code,
                            "latitude": sta.latitude,
                            "longitude": sta.longitude,
                            "elevation": sta.elevation,
                            "channels": list(set(broadband_ch))
                        })
            
            if valid_stations_info:
                radar_results.append({
                    "Event_ID": event_id,
                    "Time_UTC": str(event_time),
                    "Magnitude": row['Magnitude'],
                    "Stations": valid_stations_info
                })
            
            # SIMPAN SETIAP EVENT (Supaya benar-benar aman dari mati lampu)
            with open(OUTPUT_JSON, 'w') as f:
                json.dump(radar_results, f, indent=4)
                
        except Exception as e:
            # Jika internet putus, tnggu 5 detik lalu lanjut ke event berikutnya
            # (Agar skrip tidak mati total)
            continue

    print(f"\n✅ Radar Selesai! Data aman di: {OUTPUT_JSON}")

if __name__ == "__main__":
    run_resumable_radar()

In [ ]:
import os
import json
import time
import winsound
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm

# --- CONFIG ---
RADAR_JSON_PATH = r"D:\indonesia_output\radar_inventory_3ch_detailed.json"
SAVE_DIR = r"D:\indonesia_output\waveforms"

# Gunakan EARTHSCOPE & GFZ
client_iris = Client("EARTHSCOPE", timeout=30)
client_gfz = Client("GFZ", timeout=30)

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def start_sniper_download_minimal():
    # Load hasil radar
    if not os.path.exists(RADAR_JSON_PATH):
        print(f"❌ Error: File {RADAR_JSON_PATH} tidak ditemukan.")
        return

    with open(RADAR_JSON_PATH, 'r') as f:
        valid_events = json.load(f)

    print(f"🚀 Memulai Unduhan (Sniper Mode)")
    print(f"📦 Destinasi: {SAVE_DIR}")
    
    last_beep_time = time.time()
    
    # Progress bar tunggal (Layar tidak akan memanjang)
    pbar = tqdm(valid_events, desc="Total Progress", unit="event", dynamic_ncols=True)
    
    for event in pbar:
        event_id = event['Event_ID']
        event_time = UTCDateTime(event['Time_UTC'])
        
        # --- Heartbeat Beep (Setiap 3 detik) ---
        if time.time() - last_beep_time > 3:
            winsound.Beep(1000, 150)
            last_beep_time = time.time()

        for sta_info in event['Stations']:
            sta_code = sta_info['station']
            file_name = f"{sta_code}_{event_id}.mseed"
            file_path = os.path.join(SAVE_DIR, file_name)
            
            # Lewati jika file sudah ada (Fitur Resume)
            if os.path.exists(file_path):
                continue
            
            # Update keterangan kecil di sebelah kanan bar progres
            pbar.set_postfix_str(f"Last: {sta_code}")
                
            success = False
            for cl in [client_iris, client_gfz]:
                if success: break
                try:
                    st = cl.get_waveforms(
                        network="*", 
                        station=sta_code, 
                        location="*", 
                        channel="BH?,HH?,EH?,*?", 
                        starttime=event_time - 30, 
                        endtime=event_time + 90
                    )
                    
                    if len(st) > 0:
                        st.write(file_path, format="MSEED")
                        success = True
                except:
                    continue

    print("\n✅ Proses Selesai!")
    winsound.Beep(1500, 300)

if __name__ == "__main__":
    start_sniper_download_minimal()

🚀 Memulai Unduhan (Sniper Mode)
📦 Destinasi: D:\indonesia_output\waveforms


Total Progress:  25%|██▍       | 3380/13547 [1:29:08<7:37:37,  2.70s/event, Last: KAPI] c:\Users\Very\miniconda3\envs\waveform\lib\site-packages\obspy\io\mseed\core.py:1087: UserWarning: File will be written with more than one different record lengths.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'record lengths')
Total Progress:  47%|████▋     | 6373/13547 [2:48:46<34:52:08, 17.50s/event, Last: BOAB]c:\Users\Very\miniconda3\envs\waveform\lib\site-packages\obspy\io\mseed\core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')
Total Progress:  69%|██████▉   | 9385/13547 [20:28:12<22:53:24, 19.80s/event, Last: LUWI]  c:\Users\Very\miniconda3\envs\waveform\lib\site-packages\obspy\io\mseed\headers.py:807: InternalMSEEDWarning: GE_LUWI__BHE_D: Warning: Data integrity check for Steim2 failed, 